# TSD Fine-tuning: PhoBERT-base-v2 trên ViHOS và Đánh giá VITOSA



## 1. Cài đặt thư viện

In [1]:
!pip install -q transformers datasets accelerate huggingface_hub
!pip install -q evaluate seqeval scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


## 2. Đăng nhập Hugging Face

In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('Đăng nhập Hugging Face thành công!')
except Exception as e:
    print(f'Không tìm thấy secret: {e}')


Đăng nhập Hugging Face thành công!


## 3. Tải và khám phá ViHOS (training set)

In [3]:
import pandas as pd
from datasets import Dataset, DatasetDict

train_path = "/kaggle/input/datasets/truongpahm/datavihos/data/Sequence_labeling_based_version/Syllable/train_BIO_syllable.csv"
dev_path = "/kaggle/input/datasets/truongpahm/datavihos/data/Sequence_labeling_based_version/Syllable/dev_BIO_syllable.csv"
test_path = "/kaggle/input/datasets/truongpahm/datavihos/data/Sequence_labeling_based_version/Syllable/test_BIO_syllable.csv"

# Hàm gom dữ liệu CoNLL thành các list (câu hoàn chỉnh)
def group_conll_data(file_path):
    # Đọc file CSV
    df = pd.read_csv(file_path)
    
    # Gom nhóm theo sentence_id
    # Trả về 1 list chứa các words và 1 list chứa các tags cho mỗi câu
    grouped = df.groupby('sentence_id').agg({
        'Word': list,
        'Tag': list
    }).reset_index()
    
    # Đổi tên cột cho chuẩn với Hugging Face
    grouped = grouped.rename(columns={'Word': 'tokens', 'Tag': 'ner_tags'})
    return Dataset.from_pandas(grouped[['tokens', 'ner_tags']])

# Tạo DatasetDict hoàn chỉnh
print("Đang gom dữ liệu...")
vihos_ds = DatasetDict({
    "train": group_conll_data(train_path),
    "validation": group_conll_data(dev_path),
    "test": group_conll_data(test_path)
})

print("\n=== DỮ LIỆU ĐÃ GOM THÀNH CÂU ===")
print(vihos_ds)
print("\nMẫu câu đầu tiên:")
print("Tokens:", vihos_ds['train'][1]['tokens'])
print("Tags:  ", vihos_ds['train'][1]['ner_tags'])

Đang gom dữ liệu...

=== DỮ LIỆU ĐÃ GOM THÀNH CÂU ===
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 8844
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1106
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1106
    })
})

Mẫu câu đầu tiên:
Tokens: ['Bấp', 'bênh', 'vl', 'thế']
Tags:   ['O', 'O', 'B-T', 'O']


In [4]:
for split in vihos_ds:
    print(f'ViHOS [{split:10s}]: {len(vihos_ds[split]):,} mẫu')

ViHOS [train     ]: 8,844 mẫu
ViHOS [validation]: 1,106 mẫu
ViHOS [test      ]: 1,106 mẫu


## 4. Tải VITOSA test set (để đánh giá TSD)

In [5]:
from datasets import load_dataset

print('Đang tải file test Parquet của UIT-ViToSA/ViToSA-1.0...')

vitosa_test_set = load_dataset(
    'UIT-ViToSA/ViToSA-1.0', 
    data_files={'test': 'data/test-*.parquet'}, 
    split='test',
    verification_mode="no_checks" 
)

if 'audio' in vitosa_test_set.column_names:
    vitosa_test_set = vitosa_test_set.remove_columns(['audio'])

print("\n--- Cấu trúc Dataset thu được (Đã ẩn audio) ---")
print(vitosa_test_set)

print('\n--- Features VITOSA Test ---')
print(vitosa_test_set.features)

print('\n--- Mẫu đầu tiên VITOSA Test ---')
print(vitosa_test_set[0])

Đang tải file test Parquet của UIT-ViToSA/ViToSA-1.0...


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00003.parquet:   0%|          | 0.00/834M [00:00<?, ?B/s]

data/test-00001-of-00003.parquet:   0%|          | 0.00/833M [00:00<?, ?B/s]

data/test-00002-of-00003.parquet:   0%|          | 0.00/864M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]


--- Cấu trúc Dataset thu được (Đã ẩn audio) ---
Dataset({
    features: ['file_name', 'transcript', 'toxicity', 'split', 'label_binary_token_level', 'BIO_Tag', 'toxic_spans_text', 'toxic_spans_char_offsets'],
    num_rows: 2000
})

--- Features VITOSA Test ---
{'file_name': Value('string'), 'transcript': Value('string'), 'toxicity': Value('int64'), 'split': Value('string'), 'label_binary_token_level': Value('string'), 'BIO_Tag': Value('string'), 'toxic_spans_text': Value('string'), 'toxic_spans_char_offsets': Value('string')}

--- Mẫu đầu tiên VITOSA Test ---
{'file_name': 'test/audio4641.wav', 'transcript': 'lúc nào cũng phải trong tư thế là phải nhường tú tất cả mọi thứ phải nhường tú để cho tú địt mẹ mày xanh', 'toxicity': 1, 'split': 'test', 'label_binary_token_level': '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]', 'BIO_Tag': "['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-T', 'I-T', 'I-T

In [6]:
import ast

def convert_vitosa_direct(batch):
    outputs = {"tokens": [], "ner_tags": []}
    
    for text, bio_str in zip(batch['transcript'], batch['BIO_Tag']):
        # Cắt câu thành các từ (tokens)
        words = str(text).split() if text is not None else []
        
        # Nếu là câu sạch (None), tự tạo một mảng toàn nhãn 'O'
        if bio_str is None or str(bio_str).strip() == 'None':
            bio_tags = ['O'] * len(words)
        else:
            # Biến chuỗi "['O', 'B-T']" thành Python list thật
            try:
                bio_tags = ast.literal_eval(bio_str)
            except:
                bio_tags = ['O'] * len(words)
        
        # Đề phòng trường hợp tác giả làm data bị lệch số lượng từ và nhãn
        # Ta sẽ cắt hoặc bù nhãn 'O' cho bằng với số lượng từ
        if len(bio_tags) < len(words):
            bio_tags.extend(['O'] * (len(words) - len(bio_tags)))
        elif len(bio_tags) > len(words):
            bio_tags = bio_tags[:len(words)]

        outputs["tokens"].append(words)
        outputs["ner_tags"].append(bio_tags)
        
    return outputs

# Áp dụng cho tập VITOSA
print("Đang chuẩn hóa cấu trúc VITOSA (Dùng sẵn BIO_Tag)...")
vitosa_test_standard = vitosa_test_set.map(
    convert_vitosa_direct,
    batched=True,
    remove_columns=vitosa_test_set.column_names
)

print("\n=== MẪU VITOSA SAU KHI XỬ LÝ ===")
print("Tokens:  ", vitosa_test_standard[0]['tokens'])
print("NER Tags:", vitosa_test_standard[0]['ner_tags'])

Đang chuẩn hóa cấu trúc VITOSA (Dùng sẵn BIO_Tag)...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


=== MẪU VITOSA SAU KHI XỬ LÝ ===
Tokens:   ['lúc', 'nào', 'cũng', 'phải', 'trong', 'tư', 'thế', 'là', 'phải', 'nhường', 'tú', 'tất', 'cả', 'mọi', 'thứ', 'phải', 'nhường', 'tú', 'để', 'cho', 'tú', 'địt', 'mẹ', 'mày', 'xanh']
NER Tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-T', 'I-T', 'I-T', 'O']


## 5. Chuẩn hóa cấu trúc dataset → định dạng token classification

In [7]:
# Cấu hình Nhãn (Labels) đồng bộ cho cả ViHOS và VITOSA
LABEL_LIST = ['O', 'B-T', 'I-T']  # Bắt buộc phải dùng B-T, I-T cho khớp với data gốc
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for l, i in LABEL2ID.items()}

print('Đã cấu hình xong Labels!')
print('Mapping Labels:', LABEL2ID)

Đã cấu hình xong Labels!
Mapping Labels: {'O': 0, 'B-T': 1, 'I-T': 2}


## 6. Load tokenizer PhoBERT-base-v2

In [ ]:
import string

def normalize_text_in_dataset(example):
    """
    Tiền xử lý văn bản: viết thường và loại bỏ dấu câu.
    Lưu ý: Do là Token Classification, nếu loại bỏ 1 từ (dấu câu), 
    cần phải loại bỏ phần tử nhãn tương ứng ở cùng index.
    """
    new_tokens = []
    new_tags = []
    
    for words, tags in zip(example["tokens"], example["ner_tags"]):
        norm_words = []
        norm_tags = []
        
        for w, t in zip(words, tags):
            if pd.isna(w) or w is None:
                continue
                
            # Đưa về chữ thường và xóa hết các dấu câu xung quanh phần tử
            w_norm = str(w).lower().strip(string.punctuation)
            
            # Chỉ giữ lại những token có nội dung (không bị rỗng sau khi xóa dấu câu)
            if w_norm != '':
                norm_words.append(w_norm)
                norm_tags.append(t)
                
        new_tokens.append(norm_words)
        new_tags.append(norm_tags)
        
    return {"tokens": new_tokens, "ner_tags": new_tags}

print("Đang áp dụng normalize_text lên ViHOS...")
vihos_ds = vihos_ds.map(normalize_text_in_dataset, batched=True)

print("Đang áp dụng normalize_text lên VITOSA test...")
vitosa_test_standard = vitosa_test_standard.map(normalize_text_in_dataset, batched=True)

print("Ví dụ sau chuẩn hoá:")
print(vihos_ds['train'][1]['tokens'])
print(vihos_ds['train'][1]['ner_tags'])

In [8]:
import pandas as pd
from transformers import AutoTokenizer

MODEL_NAME = 'vinai/phobert-base-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME} | Vocab size: {tokenizer.vocab_size:,}")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: vinai/phobert-base-v2 | Vocab size: 64,000


## 7. Tokenize + align labels

In [9]:
# --- Preprocessing Function ---
def align_labels_with_tokens(examples):
    """
    Tokenizes input texts and aligns NER labels with subword tokens.
    Uses manual alignment logic as PhoBERT's slow tokenizer lacks `word_ids()` support.
    """
    tokenized_inputs = {"input_ids": [], "attention_mask": [], "labels": []}
    MAX_LEN = 256

    for words, labels in zip(examples["tokens"], examples["ner_tags"]):
        
        # Initialize with CLS token
        input_ids = [tokenizer.cls_token_id]
        label_ids = [-100]

        for word, label in zip(words, labels):
            # FIX BUG: Handle dirty data (NaN/None) from CSV
            if pd.isna(word) or word is None:
                continue

            # Ensure word is string type before tokenization
            word_str = str(word)
            word_tokens = tokenizer.tokenize(word_str)
            
            if not word_tokens:
                continue

            word_input_ids = tokenizer.convert_tokens_to_ids(word_tokens)
            
            # Safe fallback for unknown labels
            label_id = LABEL2ID.get(label, LABEL2ID['O'])

            # Subword alignment logic
            for idx, t_id in enumerate(word_input_ids):
                input_ids.append(t_id)
                
                if idx == 0:
                    label_ids.append(label_id)
                else:
                    # Propagate I-T for subsequent subwords if head is B-T
                    if label == 'B-T':
                        label_ids.append(LABEL2ID['I-T'])
                    else:
                        label_ids.append(label_id)

        # Append SEP token
        input_ids.append(tokenizer.sep_token_id)
        label_ids.append(-100)

        # Truncation & Padding handling
        if len(input_ids) > MAX_LEN:
            # Truncate while preserving the terminal SEP token
            input_ids = input_ids[:MAX_LEN-1] + [tokenizer.sep_token_id]
            label_ids = label_ids[:MAX_LEN-1] + [-100]
            attention_mask = [1] * MAX_LEN
        else:
            pad_len = MAX_LEN - len(input_ids)
            attention_mask = [1] * len(input_ids) + [0] * pad_len
            input_ids += [tokenizer.pad_token_id] * pad_len
            label_ids += [-100] * pad_len

        tokenized_inputs["input_ids"].append(input_ids)
        tokenized_inputs["attention_mask"].append(attention_mask)
        tokenized_inputs["labels"].append(label_ids)

    return tokenized_inputs

# --- 3. Execute Mapping ---
print('\nTokenizing ViHOS dataset...')
vihos_tokenized = vihos_ds.map(
    align_labels_with_tokens,
    batched=True,
    remove_columns=vihos_ds['train'].column_names 
)
print('ViHOS tokenized successfully.')

print('\nTokenizing VITOSA test dataset...')
vitosa_test_tokenized = vitosa_test_standard.map( 
    align_labels_with_tokens,
    batched=True,
    remove_columns=vitosa_test_standard.column_names
)
print('VITOSA test tokenized successfully.')


Tokenizing ViHOS dataset...


Map:   0%|          | 0/8844 [00:00<?, ? examples/s]

Map:   0%|          | 0/1106 [00:00<?, ? examples/s]

Map:   0%|          | 0/1106 [00:00<?, ? examples/s]

ViHOS tokenized successfully.

Tokenizing VITOSA test dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

VITOSA test tokenized successfully.


## 8. Load model PhoBERT-base-v2 cho Token Classification

In [ ]:
from transformers import AutoModelForTokenClassification
import torch

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
    hidden_dropout_prob=0.2,       # Tăng dropout để giảm overfitting
    attention_probs_dropout_prob=0.2
)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Model loaded. Device: {device}')
model.to(device);

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded. Device: cuda


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

## 9. Hàm tính metrics (Accuracy, WF1, MF1)

In [11]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []
    for label_row, pred_row in zip(labels, preds):
        for lbl, prd in zip(label_row, pred_row):
            if lbl != -100:
                true_labels.append(lbl)
                true_preds.append(prd)

    acc = accuracy_score(true_labels, true_preds)
    wf1 = f1_score(true_labels, true_preds, average='weighted', zero_division=0)
    mf1 = f1_score(true_labels, true_preds, average='macro',    zero_division=0)

    return {'accuracy': acc, 'weighted_f1': wf1, 'macro_f1': mf1}


def evaluate_on_dataset(trainer, tokenized_dataset, desc='Evaluating'):
    """Đánh giá model trên một dataset đã tokenize."""
    result = trainer.evaluate(eval_dataset=tokenized_dataset)
    print(f'\n--- {desc} ---')
    print(f"  Accuracy:    {result.get('eval_accuracy',   'N/A'):.4f}")
    print(f"  Weighted F1: {result.get('eval_weighted_f1','N/A'):.4f}")
    print(f"  Macro F1:    {result.get('eval_macro_f1',  'N/A'):.4f}")
    return result


print('Hàm metrics sẵn sàng')

Hàm metrics sẵn sàng


## 10. Cấu hình Training (đúng như bài báo)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir                  = './phobert-tsd-vihos',
    num_train_epochs            = 4,          
    per_device_train_batch_size = 8,          
    per_device_eval_batch_size  = 8,
    optim                       = 'adamw_torch',  
    learning_rate               = 2e-5,       
    # warmup_steps                = 500,     
    warmup_ratio                = 0.1,        
    weight_decay                = 0.01,       
    eval_strategy               = 'epoch',    
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1', 
    greater_is_better           = True,
    fp16                        = True,       
    logging_steps               = 50,
    report_to                   = 'none',
    push_to_hub                 = False,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = vihos_tokenized['train'],      
    eval_dataset    = vihos_tokenized['validation'], 
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    processing_class= tokenizer,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=1)]
)
print('  Train split:  ViHOS train')
print('  Val split:    ViHOS validation (KHÔNG ĐƯỢC GỘP để tránh overfit)')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Train split:  ViHOS train
  Val split:    ViHOS validation


## 11. Fine-tuning trên ViHOS

In [13]:
print('Bắt đầu fine-tuning PhoBERT-base-v2 trên ViHOS train set ...')
train_result = trainer.train()
print('\n Fine-tuning hoàn tất!')
print(train_result.metrics)

Bắt đầu fine-tuning PhoBERT-base-v2 trên ViHOS train set ...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Macro F1
1,0.663064,0.704023,0.873372,0.855728,0.705463
2,0.607085,0.665508,0.880385,0.871114,0.741599
3,0.439268,0.670183,0.873929,0.871618,0.750957
4,0.414843,0.697060,0.876990,0.873485,0.753781


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


 Fine-tuning hoàn tất!
{'train_runtime': 1237.5001, 'train_samples_per_second': 28.587, 'train_steps_per_second': 1.787, 'total_flos': 4621858818711552.0, 'train_loss': 0.6098479722335153, 'epoch': 4.0}


In [14]:
trainer.save_model('./phobert-tsd-best')
tokenizer.save_pretrained('./phobert-tsd-best')
print('Đã lưu model tốt nhất')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu model tốt nhất


## 12. Đánh giá trên tập TEST – ViHOS test

In [15]:
vihos_test_result = evaluate_on_dataset(
    trainer, vihos_tokenized['test'], desc='ViHOS test set'
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- ViHOS test set ---
  Accuracy:    0.8797
  Weighted F1: 0.8792
  Macro F1:    0.7548


## 13. Đánh giá trên tập TEST – VITOSA test

In [16]:
vitosa_test_result = evaluate_on_dataset(
    trainer, vitosa_test_tokenized, desc='VITOSA test set'
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- VITOSA test set ---
  Accuracy:    0.9228
  Weighted F1: 0.9307
  Macro F1:    0.7938


## 14. Tổng hợp kết quả so sánh với bài báo

In [17]:
print('=' * 60)
print('TỔNG HỢP KẾT QUẢ  –  PhoBERT-base-v2 (TSD)')
print('=' * 60)
print(f"{'Metric':<15s} | {'ViHOS test':<12s} | {'VITOSA test':<12s}")
print('-' * 60)
print(f"{'Accuracy':<15s} | {vihos_test_result.get('eval_accuracy', 0):<12.3f} | {vitosa_test_result.get('eval_accuracy', 0):<12.3f}")
print(f"{'Weighted F1':<15s} | {vihos_test_result.get('eval_weighted_f1', 0):<12.3f} | {vitosa_test_result.get('eval_weighted_f1', 0):<12.3f}")
print(f"{'Macro F1':<15s} | {vihos_test_result.get('eval_macro_f1', 0):<12.3f} | {vitosa_test_result.get('eval_macro_f1', 0):<12.3f}")
print('=' * 60)

TỔNG HỢP KẾT QUẢ  –  PhoBERT-base-v2 (TSD)
Metric          | ViHOS test   | VITOSA test 
------------------------------------------------------------
Accuracy        | 0.880        | 0.923       
Weighted F1     | 0.879        | 0.931       
Macro F1        | 0.755        | 0.794       


## 15. Suy luận định tính trên VITOSA test

In [18]:
import torch
from transformers import pipeline

# 1. Khởi tạo pipeline suy luận
ner_pipeline = pipeline(
    'token-classification',
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy='simple', # Tự động gom các subword B-T và I-T liền nhau thành cụm 'T'
    device=0 if torch.cuda.is_available() else -1
)

# 2. Lấy 5 mẫu đầu tiên từ tập VITOSA gốc để test
# Dùng tên cột cứng luôn cho an toàn
test_texts    = vitosa_test_set['transcript'][:5]
test_spans_gt = vitosa_test_set['toxic_spans_text'][:5] # Lấy cột chứa text độc hại gốc để dễ so sánh

print('Ví dụ suy luận TSD trên VITOSA test set:')
print('=' * 80)
for text, gt_text in zip(test_texts, test_spans_gt):
    if not text:
        continue
        
    # Chạy model dự đoán
    results = ner_pipeline(text)
    
    # FIX BUG: Lọc các thực thể có nhóm là 'T' (được gom từ B-T và I-T)
    toxic_ents = [r for r in results if r.get('entity_group', '') == 'T']

    print(f'Input:       {text}')
    print(f'GT Toxic:    {gt_text}') # Đáp án chuẩn của Dataset
    
    # In ra các từ model bắt được kèm độ tự tin (score)
    print(f'Pred Spans:  {[(r["word"], round(r["score"], 2)) for r in toxic_ents]}')
    print('-' * 80)

Ví dụ suy luận TSD trên VITOSA test set:
Input:       lúc nào cũng phải trong tư thế là phải nhường tú tất cả mọi thứ phải nhường tú để cho tú địt mẹ mày xanh
GT Toxic:    ['địt mẹ mày']
Pred Spans:  [('địt mẹ mày xanh', np.float32(0.95))]
--------------------------------------------------------------------------------
Input:       tại mình mình rước vào chứ mình không cho vào ai mà vào được.
GT Toxic:    None
Pred Spans:  []
--------------------------------------------------------------------------------
Input:       đụ má cái não mày cất dô cái hồ bơi hay gì vậy đụ má cho con cá bơi ở trỏng hay gì con chó đụ má mày im mẹ mồm đi
GT Toxic:    ['đụ má', 'đụ má', 'con chó đụ má mày im mẹ mồm']
Pred Spans:  [('đụ má', np.float32(0.97)), ('não mày cất dô cái', np.float32(0.74)), ('đụ má', np.float32(0.97)), ('con chó', np.float32(0.91)), ('đụ má mày', np.float32(0.84)), ('im mẹ mồm', np.float32(0.9))]
--------------------------------------------------------------------------------
Input:  

## 16. Đánh giá chi tiết: Per-label F1 trên VITOSA test

In [19]:
from sklearn.metrics import classification_report
import numpy as np
import torch
from torch.utils.data import DataLoader

# 1. Chuyển model sang chế độ đánh giá (tắt Dropout, Batch Norm...)
trainer.model.eval()

# 2. FIX LỖI CRASH: Chỉ định đích danh các cột hợp lệ cho Pytorch
valid_columns = ['input_ids', 'attention_mask', 'labels']
pytorch_dataset = vitosa_test_tokenized.select_columns(valid_columns).with_format('torch')

# 3. Tạo DataLoader
loader = DataLoader(
    pytorch_dataset,
    batch_size=16,
    collate_fn=data_collator
)

all_preds, all_labels = [], []

# 4. Bắt đầu quá trình suy luận (Inference)
print("Đang đánh giá mô hình trên tập VITOSA Test...")
with torch.no_grad():
    for batch in loader:
        # Chuyển dữ liệu lên cùng thiết bị với model (CPU hoặc GPU)
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Đưa qua model
        outputs = trainer.model(**batch)
        
        # Đưa kết quả về lại CPU và chuyển thành mảng Numpy
        logits  = outputs.logits.cpu().numpy()
        labels  = batch['labels'].cpu().numpy()

        # Lấy nhãn có xác suất cao nhất (argmax)
        pred_ids = np.argmax(logits, axis=-1)
        
        # Dàn phẳng tensor và loại bỏ các padding/special tokens (-100)
        for pred_row, label_row in zip(pred_ids, labels):
            for p, l in zip(pred_row, label_row):
                if l != -100:
                    all_preds.append(ID2LABEL[p])
                    all_labels.append(ID2LABEL[l])

# 5. In báo cáo chi tiết
print('\n=== Báo cáo Phân loại Chi tiết (Per-label Classification Report) – VITOSA TEST ===')
print(classification_report(all_labels, all_preds, labels=LABEL_LIST, digits=4))

Đang đánh giá mô hình trên tập VITOSA Test...

=== Báo cáo Phân loại Chi tiết (Per-label Classification Report) – VITOSA TEST ===
              precision    recall  f1-score   support

           O     0.9901    0.9302    0.9592     44631
         B-T     0.4621    0.8856    0.6073      2229
         I-T     0.7655    0.8711    0.8149      4756

    accuracy                         0.9228     51616
   macro avg     0.7392    0.8956    0.7938     51616
weighted avg     0.9466    0.9228    0.9307     51616



## 17. Đánh giá chi tiết: Per-label F1 trên ViHOS test

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
import torch
from torch.utils.data import DataLoader

trainer.model.eval()

# Chỉ định đích danh các cột hợp lệ cho Pytorch (để tránh lỗi crash)
valid_columns = ['input_ids', 'attention_mask', 'labels']
vihos_test_pytorch = vihos_tokenized['test'].select_columns(valid_columns).with_format('torch')

# Tạo DataLoader cho tập ViHOS test
loader_vihos = DataLoader(
    vihos_test_pytorch,
    batch_size=16,
    collate_fn=data_collator
)

all_preds_vihos, all_labels_vihos = [], []

print("Đang đánh giá chi tiết mô hình trên tập ViHOS Test...")
with torch.no_grad():
    for batch in loader_vihos:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = trainer.model(**batch)
        logits  = outputs.logits.cpu().numpy()
        labels  = batch['labels'].cpu().numpy()

        pred_ids = np.argmax(logits, axis=-1)
        
        for pred_row, label_row in zip(pred_ids, labels):
            for p, l in zip(pred_row, label_row):
                if l != -100:
                    all_preds_vihos.append(ID2LABEL[p])
                    all_labels_vihos.append(ID2LABEL[l])

# In báo cáo chi tiết cho ViHOS
print('\n=== Báo cáo Phân loại Chi tiết (Per-label Classification Report) – ViHOS TEST ===')
print(classification_report(all_labels_vihos, all_preds_vihos, labels=LABEL_LIST, digits=4))